In [24]:
import os
import pandas as pd
from pathlib import Path
from PIL import Image

In [25]:
USER_ROOT = Path("/Users/xnor/Projects/Deep-Learning/ewrd/")
DATA_ROOT = USER_ROOT / "data/raw/AnnotatedDataSet"
VALID_EXTENSIONS = ['.png', '.jpg', '.jpeg', '.tif', '.tiff', '.bmp']

In [26]:

def generate_metadata(root_dir, label_map):

    # Get the classes from the folders
    classes = [d.name for d in DATA_ROOT.iterdir() if d.is_dir()]
    print('Classes:', classes)

    rows = []
    for cls in classes:
        class_dir = DATA_ROOT / cls

        assert cls in label_map, f"Class {cls} not found in label map"
        label = label_map[cls]

        if label == -1:
            continue

        for file in class_dir.iterdir():
            if not file.suffix in VALID_EXTENSIONS:
                print(f"Warning: invalid file extension: {file}")
                continue
            fpath = file.as_posix()
            fname = file.name
            extension = file.suffix

            slide_id = "UNKNOWN"
            patient_id = "UNKNOWN"
            if "Z-" in fname and ".tif" in fname:
                try:
                    slide_id = fname.split("Z-")[1].split(".tif")[0]
                    # Extract patient_id from slide_id
                    patient_id = "-".join(slide_id.split("-")[0:2])
                except Exception:
                    slide_id = fname  # fallback
            else:
                slide_id = fname  # fallback if pattern different

            # Get image dimensions (height, width)
            try:
                with Image.open(fpath) as img:
                    width, height = img.size
            except Exception as e:
                print(f"Warning: Failed to get size for {fpath}: {e}")
                width, height = None, None

            # remove the USER_ROOT from the filepath
            fpath = str(fpath).replace(str(USER_ROOT), "")

            rows.append({
                "filepath": fpath,
                "fname": fname,
                "class_name": cls,
                "label": label,
                "slide_id": slide_id,
                'patient_id': patient_id,
                "width": width,
                "height": height
            })

    df = pd.DataFrame(rows)
    print("Total images:", len(df))
    print(df["label"].value_counts().rename(index={0:"Benign",1:"Susp/Malig"}))
    
    return df


In [27]:
import re

def assign_split(df, test_ratio=0.1, val_ratio=0.2):
    """
    Assigns splits to the dataframe based on the split ratio.
    """
    # Shuffle and split unique slide_ids to avoid leakage between splits
    #slide_ids = df['slide_id'].unique()
    #slide_ids = pd.Series(slide_ids).sample(frac=1, random_state=42)  # shuffle slide_ids

    patient_ids = df['patient_id'].unique()
    patient_ids = pd.Series(patient_ids).sample(frac=1, random_state=42)  # shuffle patient_ids

    key_ids = patient_ids

    n_unique = len(key_ids)
    train_ratio = 1 - test_ratio - val_ratio    
    n_train = int(train_ratio * n_unique)
    n_val = int(val_ratio * n_unique)
    n_test = n_unique - n_train - n_val

    #train_slide_ids = set(slide_ids.iloc[:n_train])
    #test_slide_ids = set(slide_ids.iloc[n_train:n_train+n_test])
    #val_slide_ids = set(slide_ids.iloc[n_train+n_test:])

    train_slide_ids = set(patient_ids.iloc[:n_train])
    test_slide_ids = set(patient_ids.iloc[n_train:n_train+n_test])
    val_slide_ids = set(patient_ids.iloc[n_train+n_test:])

    df['split'] = df['patient_id'].apply(lambda x: 'train' if x in train_slide_ids else 'test' if x in test_slide_ids else 'val')

    return df

def parse_ids(fname):
    slide_id = "UNKNOWN"
    patient_id = "UNKNOWN"
    patch_id = "UNKNOWN"

    # Old format: ...Z-<slide_id>.tif
    if "Z-" in fname and ".tif" in fname:
        try:
            slide_id = fname.split("Z-")[1].split(".tif")[0]
            patient_id = "-".join(slide_id.split("-")[0:2])
        except Exception:
            slide_id = fname

    # New format: S<N>-P<K>
    else:
        m = re.search(r"(S\d+)-(P\d+)", fname)
        if m:
            slide_id = m.group(1)
            patch_id = m.group(2)
            patient_id = slide_id  # or keep UNKNOWN if you prefer
        else:
            slide_id = fname

    return slide_id, patient_id, patch_id
def generate_metadata(root_dir, label_map):

    classes = [d.name for d in DATA_ROOT.iterdir() if d.is_dir()]
    print('Classes:', classes)

    rows = []
    for cls in classes:
        class_dir = DATA_ROOT / cls

        assert cls in label_map, f"Class {cls} not found in label map"
        label = label_map[cls]

        if label == -1:
            continue

        for file in class_dir.iterdir():
            if not file.suffix in VALID_EXTENSIONS:
                print(f"Warning: invalid file extension: {file}")
                continue

            fpath = file.as_posix()
            fname = file.name

            slide_id, patient_id, patch_id = parse_ids(fname)

            try:
                with Image.open(fpath) as img:
                    width, height = img.size
            except Exception as e:
                print(f"Warning: Failed to get size for {fpath}: {e}")
                width, height = None, None

            fpath = str(fpath).replace(str(USER_ROOT), "")

            rows.append({
                "filepath": fpath,
                "fname": fname,
                "class_name": cls,
                "label": label,
                "slide_id": slide_id,
                "patch_id": patch_id,
                "patient_id": patient_id,
                "width": width,
                "height": height
            })

    df = pd.DataFrame(rows)
    print("Total images:", len(df))
    print(df["label"].value_counts().rename(index={0:"Benign",1:"Susp/Malig"}))

    return df


In [28]:
label_codes_one = {'Malignant': 1, 'Benign': 0, 'Suspicious': 1}
label_codes_two = {'Malignant': 1, 'Benign': 0, 'Suspicious': -1}

df = generate_metadata(
    DATA_ROOT,
    label_codes_two
)
df = assign_split(df)
df.to_csv('/Users/xnor/Projects/Deep-Learning/ewrd/data/metadata/metadata.csv')

Classes: ['Benign', 'Malignant']
Total images: 212
label
Susp/Malig    113
Benign         99
Name: count, dtype: int64


In [29]:
df = pd.read_csv('/Users/xnor/Projects/Deep-Learning/ewrd/data/metadata/metadata.csv')

In [30]:
# Generate the Patient ID from slide_id in a vectorized and robust manner
df['patient_id'] = df['slide_id'].apply(lambda x: '-'.join(str(x).split('-')[:2]) if pd.notnull(x) else None)
df = assign_split(df)
df.to_csv('/Users/xnor/Projects/Deep-Learning/ewrd/data/metadata/metadata.csv')


In [31]:
df.head()

,Unnamed: 0,filepath,fname,class_name,label,slide_id,patch_id,patient_id,width,height,split
0,0,/Users/xnor/Projects/Deep-Learning/ewrd/data/r...,B-1.jpg,Benign,0,B-1.jpg,UNKNOWN,B-1.jpg,2048,1536,train
1,1,/Users/xnor/Projects/Deep-Learning/ewrd/data/r...,B-10.jpg,Benign,0,B-10.jpg,UNKNOWN,B-10.jpg,2048,1536,val
2,2,/Users/xnor/Projects/Deep-Learning/ewrd/data/r...,B-11.jpg,Benign,0,B-11.jpg,UNKNOWN,B-11.jpg,2048,1536,train
3,3,/Users/xnor/Projects/Deep-Learning/ewrd/data/r...,B-12.jpg,Benign,0,B-12.jpg,UNKNOWN,B-12.jpg,2048,1536,train
4,4,/Users/xnor/Projects/Deep-Learning/ewrd/data/r...,B-13.jpg,Benign,0,B-13.jpg,UNKNOWN,B-13.jpg,2048,1536,train


In [32]:
df['slide_id'].unique()

<StringArray>
[ 'B-1.jpg', 'B-10.jpg', 'B-11.jpg', 'B-12.jpg', 'B-13.jpg', 'B-14.jpg',
 'B-15.jpg', 'B-16.jpg', 'B-17.jpg', 'B-18.jpg',
 ...
 'M-90.jpg', 'M-91.jpg', 'M-92.jpg', 'M-93.jpg', 'M-94.jpg', 'M-95.jpg',
 'M-96.jpg', 'M-97.jpg', 'M-98.jpg', 'M-99.jpg']
Length: 212, dtype: str

In [33]:
df.groupby(['patient_id', 'split']).count()

,,Unnamed: 0,filepath,fname,class_name,label,slide_id,patch_id,width,height
patient_id,split,,,,,,,,,
B-1.jpg,train,1,1,1,1,1,1,1,1,1
B-10.jpg,val,1,1,1,1,1,1,1,1,1
B-11.jpg,train,1,1,1,1,1,1,1,1,1
B-12.jpg,train,1,1,1,1,1,1,1,1,1
B-13.jpg,train,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...
M-95.jpg,train,1,1,1,1,1,1,1,1,1
M-96.jpg,val,1,1,1,1,1,1,1,1,1
M-97.jpg,train,1,1,1,1,1,1,1,1,1


In [34]:
df.groupby(['split']).count()

,Unnamed: 0,filepath,fname,class_name,label,slide_id,patch_id,patient_id,width,height
split,,,,,,,,,,
test,22,22,22,22,22,22,22,22,22,22
train,148,148,148,148,148,148,148,148,148,148
val,42,42,42,42,42,42,42,42,42,42
